# Setup
データ取り込みを行う。 DuckDB 想定。

# Extract

In [ ]:
from pathlib import Path
from assistant_agent.loaders import ObsidianReader

# Vault から読み込み
vault_path = Path("../../docs/dataset_obsidian/")
loader = ObsidianReader(vault_path)
docs = loader.load_data()

print(f"{len(docs)} 件のノートを読み込みました")

# Load

In [ ]:
import os
import dotenv
from pathlib import Path

dotenv.load_dotenv()
ENV_GEMINI_API_KEY = os.getenv("ENV_GEMINI_API_KEY")

# 保存先ディレクトリを作成
vault_db_path = Path("../../tests/data/vault_db").resolve()
vault_db_path.mkdir(exist_ok=True)
vault_db_path = vault_db_path.resolve()

In [ ]:
from pprint import pprint

from sqlalchemy import create_engine, text
from assistant_agent.entities.duckdb import VaultBase, ObsidianEntity
from assistant_agent.entities.base import VaultUtils

path = vault_db_path / "entity.duckdb"
sa_engine = create_engine(f"duckdb:///{path}")

# DB へ取り込み
with sa_engine.connect() as sess:
    sess.execute(text("create schema if not exists assets;"))
    sess.commit()

VaultBase.metadata.create_all(sa_engine)
VaultUtils.sync(docs, sa_engine, ObsidianEntity)
print(f"{len(docs)} 件のノートを読み込みました")

with sa_engine.connect() as sess:
    # sess.execute(text("checkpoint"))
    res = sess.execute(text("select * from entity.assets.obsidian_raw limit 3"))
    pprint(res.all())

sa_engine.dispose()

In [ ]:
from llama_index.core.ingestion import IngestionPipeline
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.schema import TransformComponent

chunk_size = 1024
chunk_overlap = 200
JAPANESE_PARAGRAPH_SEP = "\n\n"

trans: list[TransformComponent] = [
    SentenceSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        paragraph_separator=JAPANESE_PARAGRAPH_SEP,
    ),
]
pipe = IngestionPipeline(transformations=trans)
res = pipe.run(documents=docs)
print("\n=====================\n".join([item.text for item in res]))  # pyright: ignore[reportAttributeAccessIssue]


In [ ]:
from llama_index.core.node_parser import SentenceSplitter
from llama_index.embeddings.google_genai import GoogleGenAIEmbedding

from assistant_agent.entities.duckdb import ObsidianEntity
from assistant_agent.services import VaultObsidianRetriever
from assistant_agent.store import DuckDBStoreConnector

# レトリーバーを作成
store_conn = DuckDBStoreConnector(str(vault_db_path))
sa_engine = store_conn.get_engine()
obsidian_retriever = VaultObsidianRetriever(
    "obsidian_docstore",
    "obsidian_vectors",
    store_conn=store_conn,
    transformations=[
        SentenceSplitter(
            chunk_size=1024,
            chunk_overlap=200,
            paragraph_separator="\n\n",
        ),
    ],
    embed_model=GoogleGenAIEmbedding(
        model_name="gemini-embedding-001",
        api_key=ENV_GEMINI_API_KEY,
    ),
    embed_dim=3072,
    vault_entity=ObsidianEntity,
)

In [ ]:
obsidian_retriever.sync_chunks()

# Retrieval

In [ ]:
query_res = obsidian_retriever.search_documents("コンテキストエンジニアリング", 5)
query_res = obsidian_retriever.get_documents_by_ids([item.id_ for item in query_res])
for item in query_res[:3]:
    print(item.text)
    print("==================")

In [ ]:
from llama_index.core.vector_stores.types import (
    VectorStoreQuery, VectorStoreQueryMode,
    MetadataFilters, MetadataFilter, FilterOperator
)
from llama_index.embeddings.google_genai import GoogleGenAIEmbedding


ember = GoogleGenAIEmbedding(model_name="gemini-embedding-001", api_key=os.getenv("ENV_GEMINI_API_KEY"))
query_res = obsidian_retriever._vector_store.query(
    VectorStoreQuery(
        query_embedding=ember.get_query_embedding("プロンプトエンジニアリング"),
        similarity_top_k=5,
        mode=VectorStoreQueryMode.MMR,
        mmr_threshold=0.5,
        filters=MetadataFilters(
            filters=[
                MetadataFilter(key="file_path", value="03_Structure/", operator=FilterOperator.TEXT_MATCH)
            ]
        )
    )
)
assert query_res.nodes is not None
for item in query_res.nodes:
    print(item.text)  # pyright: ignore[reportAttributeAccessIssue]
    print("==================")

In [ ]:
store_conn.close()

# ContextRegistry

In [ ]:
from typing import cast
from pathlib import Path

import assistant_agent.services  # noqa: F401  登録発火
from assistant_agent.store import DuckDBStoreConnector
from assistant_agent.tools.obsidian import ObsidianContext
from assistant_agent.utils.context import ContextRegistry

vault_db_path = Path("../../tests/data/vault_db")
vault_db_path.mkdir(exist_ok=True)
vault_db_path = vault_db_path.resolve()

cr_ctx = DuckDBStoreConnector(str(vault_db_path), read_only=True)
ctx = cast(ObsidianContext, ContextRegistry.build(store_conn=cr_ctx))

In [ ]:
ctx['obsidian_retriever'].search_documents("コンテキストエンジニアリング", top_k=3)